In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle
import numpy as np
import datetime
from IPython.display import display, HTML

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


In [ ]:
print("================================================================================")
print(" 🚀 SETUP INSERT HANDLER GLOBAL - FASE 2 (SALING SILANG & AUTO-SKIP) 🚀 ")
print("================================================================================")

# ================================================================================
# TAHAP 1: MEMUAT SE LURUH BERKAS PICKLE DAN MERGE JADI SATU PINTU
# ================================================================================
all_fase_2_data = {}

# 1. Load File Cimut
try:
    with open('fase_2_cimut.pkl', 'rb') as f:
        all_fase_2_data.update(pickle.load(f))
    print("✓ Berhasil memuat data hasil konversi Cimut.")
except Exception as e:
    print(f"⚠️ Gagal memuat file pkl Cimut: {e}")

# 2. Load File Afrida
try:
    with open('fase_2_afrida.pkl', 'rb') as f:
        all_fase_2_data.update(pickle.load(f))
    print("✓ Berhasil memuat data hasil konversi Afrida.")
except Exception as e:
    print(f"⚠️ Gagal memuat file pkl Afrida: {e}")

# 3. Load File Hanif (Jika ada file terpisah, opsional)
try:
    if os.path.exists('fase_2_hanif.pkl'):
        with open('fase_2_hanif.pkl', 'rb') as f:
            all_fase_2_data.update(pickle.load(f))
        print("✓ Berhasil memuat data hasil konversi Hanif.")
except:
    pass

 🚀 SETUP INSERT HANDLER GLOBAL - FASE 1 (SALING SILANG & AUTO-SKIP) 🚀 
✓ Berhasil memuat data hasil konversi Cimut.
✓ Berhasil memuat data hasil konversi Afrida.


In [ ]:
# ================================================================================
# TAHAP 2: ATUR URUTAN STRATEGIS SALING SILANG (KARYA CIMUT & AFRIDA)
# ================================================================================
# Susun urutannya di sini secara mutlak, bebas saling silang antar tim!
tables_to_insert_ordered = [
    # --- Blok Awal: Data Induk Fondasi (Milik Cimut) ---
    'periode','parameter_nilai', 'karyawan', 'bidang_kategori', 'keluarga_karyawan', 'bidang_link'   
]

In [ ]:
import pandas as pd
import datetime
import numpy as np
import mysql.connector

# ================================================================================
# TAHAP 3: FUNGSI UTAMA INSERT DENGAN RINGKASAN DI ATAS & DIAGNOSTIK ERROR DI BAWAH
# ================================================================================
def insert_data_with_preview_and_skip_v2(db_connection, cursor, tables_data, ordered_list):
    results = {}
    
    print("="*80)
    print("🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)")
    print("="*80)
    
    # ----------------------------------------------------------------------------
    # SUB-LANGKAH A: PROSES INSERT KE MYSQL DI BELAKANG LAYAR
    # ----------------------------------------------------------------------------
    for table_name in ordered_list:
        if table_name not in tables_data:
            results[table_name] = {
                'status': 'not_found', 
                'rows': 0, 
                'msg': f'⚠️  {table_name}: Tidak ditemukan di file pkl'
            }
            continue
            
        df_target = tables_data[table_name]
        
        if df_target is None or df_target.empty:
            results[table_name] = {
                'status': 'empty', 
                'rows': 0, 
                'msg': f'ℹ️  {table_name}: DataFrame kosong (0 baris)'
            }
            continue
            
        try:
            # Bersihkan kolom kosong murni agar tidak merusak placeholder query
            df_to_push = df_target.dropna(axis=1, how='all')
            
            columns_str = ', '.join([f'`{col}`' for col in df_to_push.columns])
            placeholders_str = ', '.join(['%s'] * len(df_to_push.columns))
            
            # Gunakan INSERT IGNORE untuk auto-skip duplikat primary key
            insert_query = f"INSERT IGNORE INTO `{table_name}` ({columns_str}) VALUES ({placeholders_str})"
            
            # Konversi DataFrame ke Native List Python (Hancurkan tipe data NumPy)
            raw_numpy_list = df_to_push.to_numpy().tolist()
            clean_data_tuples = [
                tuple(None if pd.isna(x) or str(x).strip() in ["NaT", "NaN"] else x for x in row) 
                for row in raw_numpy_list
            ]
            
            # Eksekusi massal
            cursor.executemany(insert_query, clean_data_tuples)
            db_connection.commit()
            
            total_rows = len(clean_data_tuples)
            results[table_name] = {
                'status': 'success', 
                'rows': total_rows, 
                'msg': f'✓ {table_name}: Sukses diproses! Sebanyak {total_rows} baris sukses dimasukkan / di-skip aman.'
            }
            
        except Exception as e:
            db_connection.rollback()
            results[table_name] = {
                'status': 'failed', 
                'rows': 0, 
                'msg': f'✗ {table_name}: Gagal total saat insert - Alasan: {e}'
            }

    # ----------------------------------------------------------------------------
    # 📊 CETAK PAPAN RINGKASAN DI PALING ATAS (SUMMARY BOARD)
    # ----------------------------------------------------------------------------
    print("\n================================================================================")
    print(" 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊")
    print("================================================================================")
    
    # 1. Cetak yang sukses dulu biar rapi
    print("🟢 TABEL YANG SUKSES MASUK:")
    success_exist = False
    for table_name in ordered_list:
        if table_name in results and results[table_name]['status'] == 'success':
            print(f"  {results[table_name]['msg']}")
            success_exist = True
    if not success_exist: print("  (Tidak ada tabel yang sukses)")

    print("\n🔴 TABEL YANG BERMASALAH / GAGAL MASUK (MOHON DIANALISA):")
    failed_exist = False
    for table_name in ordered_list:
        if table_name in results and results[table_name]['status'] in ['failed', 'not_found', 'empty']:
            print(f"  {results[table_name]['msg']}")
            failed_exist = True
    if not failed_exist: print("  🎉 LUAR BIASA! Semua tabel bersih tidak ada yang gagal.")
            
    print("================================================================================\n")


    # ----------------------------------------------------------------------------
    # 📸 CETAK PREVIEW HISTORI & DIAGNOSTIK ERROR DI BAGIAN BAWAH
    # ----------------------------------------------------------------------------
    print("="*80)
    print("📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL")
    print("="*80)
    
    for table_name in ordered_list:
        if table_name in results:
            # Taktik A: Jika Sukses, tampilkan preview standard 5 baris teratas
            if results[table_name]['status'] == 'success':
                print(f"\n📂 [🟢 PREVIEW TABEL SUKSES: {table_name.upper()}]")
                print("-" * 50)
                display(tables_data[table_name]) 
                print("-" * 80)
                
            # Taktik B: Jika GAGAL, tembak dan kuliti struktur datanya secara transparan!
            elif results[table_name]['status'] == 'failed':
                print(f"\n🚨 [🔴 DIAGNOSTIK TABEL ERROR: {table_name.upper()}] 🚨")
                print(f"Alasan MySQL Menolak: {results[table_name]['msg']}")
                print("-" * 50)
                print("Berikut 5 baris sampel data yang gagal dikirim, cek tipe datanya (apakah ada .0 atau string aneh):")
                display(tables_data[table_name])
                print(f"\nTipe data kolom internal DataFrame untuk tabel '{table_name}':")
                # Menampilkan tipe data internal pandas agar ketahuan mana float ghaib / objek aneh
                print(tables_data[table_name].dtypes)
                print("-" * 80)
            
    print("\n" + "="*80)
    print("🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁")
    print("="*80)
    return results

In [ ]:
# ================================================================================
# TAHAP 4: MENJALANKAN EKSEKUSI DATA REAL (MENGGUNAKAN VERSI RINGKASAN ATAS)
# ================================================================================
results_fase_2 = insert_data_with_preview_and_skip_v2(
    db_connection=db_new, 
    cursor=cursor_new, 
    tables_data=all_fase_2_data, 
    ordered_list=tables_to_insert_ordered
)

🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)

 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊
✓ users: Sukses diproses! Sebanyak 51 baris sukses dimasukkan / di-skip aman.
✓ divisions: Sukses diproses! Sebanyak 6 baris sukses dimasukkan / di-skip aman.
✓ shift_kerja: Sukses diproses! Sebanyak 3 baris sukses dimasukkan / di-skip aman.
✓ kursus: Sukses diproses! Sebanyak 21 baris sukses dimasukkan / di-skip aman.
✓ level: Sukses diproses! Sebanyak 181 baris sukses dimasukkan / di-skip aman.
✓ sesi: Sukses diproses! Sebanyak 43 baris sukses dimasukkan / di-skip aman.
✓ libur: Sukses diproses! Sebanyak 79 baris sukses dimasukkan / di-skip aman.
✓ admin_sarpras: Sukses diproses! Sebanyak 1 baris sukses dimasukkan / di-skip aman.
✓ sop_kategori: Sukses diproses! Sebanyak 3 baris sukses dimasukkan / di-skip aman.
✓ topik_diskusi: Sukses diproses! Sebanyak 11 baris sukses dimasukkan / di-skip aman.
✓ kursus_libur: Sukses diproses! Sebanyak 2 baris sukses

,id_user,name,email,email_verified_at,password,remember_token,created_at,updated_at
0,U00001,ADMINISTRATOR,ditari@leapsurabaya.sch.id,NaN,aGtq,NaN,NaN,NaN
1,U00003,"Graciela Evanda Ronadi, S.Kom.",graciela@leapsurabaya.sch.id,NaN,qWmlbcVjYmo%3D,NaN,NaN,NaN
2,U00011,DANIAR AULIA RIZKI,daniar.rizki@leapsurabaya.sch.id,NaN,aGtq,NaN,NaN,NaN
3,U00012,Habibah Melyna,habibah.elfiani@leapsurabaya.sch.id,NaN,o56YqZJkZA%3D%3D,NaN,NaN,NaN
4,U00014,Laksmi Puspitowardhani,laksmi.p@leapsurabaya.sch.id,NaN,aWlpbw%3D%3D,NaN,NaN,NaN


--------------------------------------------------------------------------------

📂 [PREVIEW TABEL: DIVISIONS]
--------------------------------------------------


,id_division,name_division,description,is_active
0,NaN,IT,None,1
1,NaN,Busdev,None,1
2,NaN,HR / GA,None,1
3,NaN,Pendidikan,None,1
4,NaN,Finance,None,1


--------------------------------------------------------------------------------

📂 [PREVIEW TABEL: SHIFT_KERJA]
--------------------------------------------------


...

--------------------------------------------------------------------------------

📂 [PREVIEW TABEL: KURSUS_LIBUR]
--------------------------------------------------


,id_kursus,id_libur
0,K00001,L00070
1,K00001,L00068


--------------------------------------------------------------------------------

📂 [PREVIEW TABEL: KURSUS_LEVEL]
--------------------------------------------------


,id_kursus,id_level
0,K00001,L00001
1,K00001,L00002
2,K00001,L00003
3,K00001,L00004
4,K00001,L00005


--------------------------------------------------------------------------------

🏁 SELURUH DAFTAR TABEL FASE 1 SUKSES DI-PUSH DENGAN SKEMA SALING SILANG 🏁


In [ ]:
# print("================================================================================")
# print(" 🧹 MEMULAI PROSES TRUNCATE DATA GLOBAL - FASE 2 (SISTEM RINGKASAN ATAS) 🧹 ")
# print("================================================================================")

# def truncate_tables_with_summary(db_connection, cursor, ordered_list):
#     truncate_results = {}
    
#     try:
#         # 🔥 SAKTI 1: Matikan benteng Foreign Key checks agar MySQL tidak memblokir penghapusan
#         cursor.execute("SET FOREIGN_KEY_CHECKS=0")
#         db_connection.commit()
#         print("🔓 Sensor Foreign Key Checks berhasil DIMATIKAN sementara.\n")
#     except Exception as e:
#         print(f"✗ Gagal mematikan Foreign Key Checks: {e}")
#         return
        
#     # ----------------------------------------------------------------------------
#     # SUB-LANGKAH A: PROSES EKSEKUSI TRUNCATE DI BELAKANG LAYAR
#     # ----------------------------------------------------------------------------
#     for table_name in ordered_list:
#         try:
#             truncate_query = f"TRUNCATE TABLE `{table_name}`"
#             cursor.execute(truncate_query)
#             db_connection.commit()
            
#             truncate_results[table_name] = {
#                 'status': 'success',
#                 'msg': f"✓ {table_name}: Sukses dibersihkan total! Seluruh baris data amblas."
#             }
#         except Exception as e:
#             db_connection.rollback()
#             truncate_results[table_name] = {
#                 'status': 'failed',
#                 'msg': f"✗ {table_name}: Gagal dikosongkan! Alasan: {e}"
#             }

#     try:
#         # 🔥 SAKTI 2: Wajib nyalakan kembali benteng Foreign Key checks setelah selesai
#         cursor.execute("SET FOREIGN_KEY_CHECKS=1")
#         db_connection.commit()
#         print("🔒 Sensor Foreign Key Checks berhasil DIHIDUPKAN kembali dengan aman.")
#     except Exception as e:
#         print(f"⚠️ Peringatan: Gagal menghidupkan kembali Foreign Key Checks: {e}")

#     # ----------------------------------------------------------------------------
#     # 🔥 CETAK PAPAN RINGKASAN TRUNCATE DI PALING ATAS (ANTI-SCROLL BOARD)
#     # ----------------------------------------------------------------------------
#     print("\n================================================================================")
#     print(" 📊 PAPAN RINGKASAN STATUS TRUNCATE DATABASE (CLEANUP SUMMARY BOARD) 📊")
#     print("================================================================================")
#     for table_name in ordered_list:
#         if table_name in truncate_results:
#             print(truncate_results[table_name]['msg'])
#         else:
#             print(f"⚠️  {table_name}: Lewat dari antrean pembersihan.")
#     print("================================================================================")
    
#     return truncate_results

# # === JALANKAN EKSEKUSI PEMBERSIHAN MENGGUNAKAN DAFTAR TABEL SALING SILANGMU ===
# results_truncate_fase_2 = truncate_tables_with_summary(
#     db_connection=db_new, 
#     cursor=cursor_new, 
#     ordered_list=tables_to_insert_ordered  # Otomatis memakai list urutan saling silang yang kita buat tadi
# )

 🧹 MEMULAI PROSES TRUNCATE DATA GLOBAL - FASE 1 (SISTEM RINGKASAN ATAS) 🧹 
🔓 Sensor Foreign Key Checks berhasil DIMATIKAN sementara.



🔒 Sensor Foreign Key Checks berhasil DIHIDUPKAN kembali dengan aman.

 📊 PAPAN RINGKASAN STATUS TRUNCATE DATABASE (CLEANUP SUMMARY BOARD) 📊
✓ users: Sukses dibersihkan total! Seluruh baris data amblas.
✓ divisions: Sukses dibersihkan total! Seluruh baris data amblas.
✓ shift_kerja: Sukses dibersihkan total! Seluruh baris data amblas.
✓ kursus: Sukses dibersihkan total! Seluruh baris data amblas.
✓ level: Sukses dibersihkan total! Seluruh baris data amblas.
✓ sesi: Sukses dibersihkan total! Seluruh baris data amblas.
✓ libur: Sukses dibersihkan total! Seluruh baris data amblas.
✓ admin_sarpras: Sukses dibersihkan total! Seluruh baris data amblas.
✓ sop_kategori: Sukses dibersihkan total! Seluruh baris data amblas.
✓ topik_diskusi: Sukses dibersihkan total! Seluruh baris data amblas.
✓ kursus_libur: Sukses dibersihkan total! Seluruh baris data amblas.
✓ kursus_level: Sukses dibersihkan total! Seluruh baris data amblas.
